# Práctica 2: Aprendizaje No Supervisado
## Determinación de Tipos de Estrellas

**Asignatura:** Aprendizaje Automático  
**Dataset:** stars_data.csv — 240 estrellas con atributos físicos y espectrales

El objetivo es aplicar técnicas de clustering (K-Means, Hierarchical Clustering y DBSCAN)
para identificar tipos de estrellas de forma no supervisada.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage

SEED = 100522187

np.random.seed(SEED)

## 1. Carga y Exploración de los Datos

In [ ]:
#cargamos el dataset
dataframe = pd.read_csv('stars_data.csv')

#mostramos las primeras filas
print("Primeras filas del dataset:")
dataframe.head()

In [ ]:
#información general: tipos de datos y valores nulos
print("Información del dataset:")
dataframe.info()

print("\nValores nulos por columna:")
print(dataframe.isnull().sum())

## 2. Preprocesamiento: Codificación Ordinal de Variables Categóricas

Las variables `Color` y `Spectral_Class` son categóricas pero tienen un orden físico:
- **Spectral_Class**: de O (más caliente) a M (más fría) → O=6, B=5, A=4, F=3, G=2, K=1, M=0
- **Color**: de mayor a menor energía → azul > blanco > amarillo > naranja > rojo

In [ ]:
#codificación ordinal de Spectral_Class (
spectral_order = {'O': 6, 'B': 5, 'A': 4, 'F': 3, 'G': 2, 'K': 1, 'M': 0}
dataframe['Spectral_Class_enc'] = dataframe['Spectral_Class'].map(spectral_order)

#codificación ordinal de Color (de mayor a menor energía/temperatura)
color_order = {
    'Blue': 7,
    'Blue-white': 6,
    'Blue White': 6,
    'Blue white': 6,
    'Blue-White': 6,
    'White': 5,
    'yellow-white': 4,
    'Yellow-white': 4,
    'White-Yellow': 4,
    'Yellowish White': 4,
    'white': 5,
    'Yellowish': 3,
    'Whitish': 5,
    'Orange': 2,
    'Orange-Red': 1,
    'Red': 0,
    'yellowish': 3,
    'Pale yellow orange': 2
}
dataframe['Color_enc'] = dataframe['Color'].map(color_order)

#verificamos valores no mapeados
print("NaN en Spectral_Class_enc:", dataframe['Spectral_Class_enc'].isnull().sum())
print("NaN en Color_enc:", dataframe['Color_enc'].isnull().sum())

#si hay NaN, significa que hay algún valor de color no contemplado en el diccionario
if dataframe['Color_enc'].isnull().sum() > 0:
    print("Valores no mapeados:", dataframe[dataframe['Color_enc'].isnull()]['Color'].unique())

## 3. Normalización y Reducción de Dimensionalidad con PCA

Antes de aplicar PCA normalizamos los datos con StandardScaler para que todas las 
variables tengan la misma escala. Luego reducimos a 2 componentes principales para 
facilitar la visualización y el clustering.

In [ ]:
#seleccionamos las features para el clustering
features = ['Temperature', 'L', 'R', 'A_M', 'Spectral_Class_enc', 'Color_enc']
X = dataframe[features].copy()

#normalizamos los datos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

#aplicamos PCA con 2 componentes
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)

print("Varianza explicada por cada componente:", pca.explained_variance_ratio_)
print(f"Varianza total explicada: {pca.explained_variance_ratio_.sum():.2%}")

In [ ]:
#visualizamos los datos en el espacio de 2 componentes principales
plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.6, edgecolors='k', linewidths=0.3)
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.title('Datos proyectados en 2 componentes principales (PCA)')
plt.tight_layout()
plt.show()

## 4. Clustering

### 4.1 K-Means

Usamos el **método del codo** (inercia) y la **Silhouette Score** para determinar 
el número óptimo de clusters k.

In [ ]:
inertias = []
silhouettes = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_pca, labels))

#grafica el método del codo y silhouette score
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias, marker='o')
axes[0].set_title('Método del Codo')
axes[0].set_xlabel('Número de clusters (k)')
axes[0].set_ylabel('Inercia')

axes[1].plot(k_range, silhouettes, marker='o', color='orange')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Número de clusters (k)')
axes[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

In [ ]:
#seleccionamos el k óptimo según las gráficas anteriores
k_optimo = 6  

km_final = KMeans(n_clusters=k_optimo, random_state=SEED, n_init=10)
labels_km = km_final.fit_predict(X_pca)

#visualizamos los clusters
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_km, cmap='tab10', 
                      alpha=0.7, edgecolors='k', linewidths=0.3)
plt.colorbar(scatter, label='Cluster')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.title(f'K-Means con k={k_optimo}')
plt.tight_layout()
plt.show()

print(f"Silhouette Score (K-Means, k={k_optimo}): {silhouette_score(X_pca, labels_km):.4f}")
print(f"Davies-Bouldin Score (menor es mejor): {davies_bouldin_score(X_pca, labels_km):.4f}")

# --- Celda para visualizar el Dendrograma ---
plt.figure(figsize=(10, 6))
plt.title('Dendrograma Jerárquico (Linkage: Ward)')
plt.xlabel('Índices de las muestras')
plt.ylabel('Distancia')

# Generamos la matriz de enlace usando el método ward
Z = linkage(X_pca, method='ward')
dendrogram(Z, truncate_mode='level', p=5) # Truncamos para que sea más legible
plt.axhline(y=15, color='r', linestyle='--') # Línea de corte de ejemplo
plt.tight_layout()
plt.show()

# --- Celda para aplicar Agglomerative Clustering ---
# Puedes cambiar 'ward' por 'average' o 'complete' si lo justificas en base al dendrograma
hc = AgglomerativeClustering(n_clusters=6, linkage='ward')
labels_hc = hc.fit_predict(X_pca)

# Visualizamos los clusters
plt.figure(figsize=(8, 6))
scatter_hc = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_hc, cmap='tab10', 
                         alpha=0.7, edgecolors='k', linewidths=0.3)
plt.colorbar(scatter_hc, label='Cluster')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.title('Hierarchical Clustering (Ward, k=6)')
plt.tight_layout()
plt.show()

# Evaluamos con las métricas clásicas
print(f"Silhouette Score (Hierarchical, k=6): {silhouette_score(X_pca, labels_hc):.4f}")
print(f"Davies-Bouldin Score (menor es mejor): {davies_bouldin_score(X_pca, labels_hc):.4f}")

# --- Celda para ajuste de hiperparámetros de DBSCAN ---
from dbcv import dbcv # Asegúrate de haber instalado la librería

eps_values = np.arange(0.2, 1.0, 0.1)
min_samples_values = range(3, 10)

best_dbcv = -1
best_params = {'eps': None, 'min_samples': None}
best_labels = None

for eps in eps_values:
    for min_samples in min_samples_values:
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        labels = dbscan.fit_predict(X_pca)
        
        # Ignoramos configuraciones que catalogan todo como ruido (-1) o hacen un solo cluster
        if len(set(labels)) > 1 and sum(labels == -1) / len(labels) < 0.5: 
            try:
                # DBCV devuelve un valor entre -1 y 1 (mayor es mejor)
                score = dbcv(X_pca, labels)
                if score > best_dbcv:
                    best_dbcv = score
                    best_params = {'eps': eps, 'min_samples': min_samples}
                    best_labels = labels
            except ValueError:
                # DBCV puede fallar si la forma de los clusters es muy anómala en ciertas iteraciones
                continue

print(f"Mejores parámetros DBSCAN: {best_params}")
print(f"Mejor score DBCV: {best_dbcv:.4f}")

# Visualizamos el mejor resultado de DBSCAN
plt.figure(figsize=(8, 6))
# El cluster -1 es ruido en DBSCAN
scatter_db = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=best_labels, cmap='tab10', 
                         alpha=0.7, edgecolors='k', linewidths=0.3)
plt.colorbar(scatter_db, label='Cluster (-1 = Ruido)')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.title(f"DBSCAN (eps={best_params['eps']:.2f}, min_samples={best_params['min_samples']})")
plt.tight_layout()
plt.show()